In [ ]:
%pip install --upgrade --quiet  langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 4.2 MB/s eta 0:00:00


In [ ]:
import os
import getpass
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser
from langsmith import traceable
from pydantic import BaseModel, Field
from typing import List

In [ ]:
if "LANGSMITH_API_KEY" not in os.environ:
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API key: ")

llm = ChatOpenAI(model="gpt-4o", temperature=0)

class ResumeData(BaseModel):
    name: str = Field(description="Full name of the person")
    email: str = Field(description="Email address of the person")
    phone: str = Field(description="Phone number of the person")
    skills: List[str] = Field(description="List of key skills mentioned")

parser = PydanticOutputParser(pydantic_object=ResumeData)

prompt = PromptTemplate(
    template="""
Extract structured information from the following resume text.
Return in JSON format.

Resume Text:
{resume}

{format_instructions}
""",
    input_variables=["resume"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

@traceable
def extract_resume_data(resume_text: str) -> ResumeData:
    chain = prompt | llm | parser
    return chain.invoke({"resume": resume_text})